Notebook de pruebas:
Tiene todas las funcionalidades del standalone (main_despliegue_standalone)

Nota: Si este notebook da errores de importación asociados a Cartopy (CRSS); se debe reiniciar el kernel y volver a ejecutar todo

In [1]:
####################### N O  T O C A R ############################################
%reload_ext autoreload
%autoreload 2

import os
import sys
import copy

import pandas as pd

root_path = os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), ".."))
sys.path.append(root_path)

# Importar configuraciones del modulo
from procesado_datos.config_modulo.config_procesado import config_modulo_procesado
# Importar configuraciones del submodulo de despliegue
from procesado_datos.transmision.configs.configuracion_transmision import config_transmision

# Importar manager de configuraciones
from procesado_datos.config_modulo.ProcesadoConfig import ProcesadoConfig

# Importar servicios necesarios
from pathlib import Path
import procesado_datos.services.Carga.cargar_datos_csv as carga
from procesado_datos.services.Utils.utilidades import *
from procesado_datos.services.Correctores.corrector_utils import *
from procesado_datos.services.Graficado.graficar_series_y_guardar import graficar_series_y_guardar
from procesado_datos.services.Graficado.graficar_mapa_de_posiciones import graficar_mapa_de_posiciones
from procesado_datos.services.Utils.excel_a_png import *
from procesado_datos.transmision.configs import configuracion_transmision

##################################################################################

In [2]:
# Crear la instancia del manager de configuraciones
config = ProcesadoConfig.from_sources(
    config_modulo_procesado,
    config_transmision,
)    

In [3]:
config.variables_a_graficar

['temperatura_mar',
 'u_corriente',
 'v_corriente',
 'rap_corriente',
 'dir_corriente',
 'voltaje']

In [4]:
# Crear ruta a la carpeta de guardado de datos
ruta_a_carpeta = config.carpeta_de_guardado_de_datos_procesados 
fecha_del_estudio = config.convertir_a_pd_datetime("fecha_del_estudio", formato="%Y-%m-%d")
carpeta_del_estudio = f"{fecha_del_estudio.year:04d}{fecha_del_estudio.month:02d}"
ruta_a_la_carpeta_de_guardado = os.path.join(ruta_a_carpeta, carpeta_del_estudio)
ruta_a_la_carpeta_de_guardado

'C:\\Users\\Atmosfera\\Desktop\\datos_procesados\\doris\\10.3\\despliegue_2\\202610'

In [5]:
# 3
rutas_de_sondas, seriales_encontrados = carga.buscar_nombre_de_archivo_de_sonda(config)

In [6]:
# 4
diccionario_de_datos_de_sondas = carga.cargar_datos_de_sonda(rutas_de_sondas, seriales_encontrados)

Datos cargados correctamente para la sonda: B048-9869012
Datos cargados correctamente para la sonda: B072-5339652
Datos cargados correctamente para la sonda: B071-5339826
Datos cargados correctamente para la sonda: B073-5339822
Datos cargados correctamente para la sonda: B079-5339955


In [7]:
# 5.
diccionario_de_datos_de_sondas = reemplazar_datos_de_rapidez_distancia_y_direccion_de_primera_medicion_por_null(diccionario_de_datos_de_sondas)

Sonda B048-9869012: Se eliminaron los datos de rapidez, distancia y dirección de la primera medición.
Sonda B072-5339652: Se eliminaron los datos de rapidez, distancia y dirección de la primera medición.
Sonda B071-5339826: Se eliminaron los datos de rapidez, distancia y dirección de la primera medición.
Sonda B073-5339822: Se eliminaron los datos de rapidez, distancia y dirección de la primera medición.
Sonda B079-5339955: Se eliminaron los datos de rapidez, distancia y dirección de la primera medición.


In [8]:
# diccionario_de_datos_de_sondas["4912205"]

In [9]:
# Convertir los datos a tipo string y reemplazar 'nan' por 'null' en la primera fila de cada DataFrame
nombres_de_columnas_de_pagina_web = ["fecha", "latitud", "logitud", "speed", "distance", "direction", "direction_gen", "temp", "volt"]
dict_guardado = copy.deepcopy(diccionario_de_datos_de_sondas)
for key, df in dict_guardado.items():
    for col in df.columns:
        if col in ["rap_corriente", "distancia", "dir_corriente","dir_corriente_texto"]:
            dict_guardado[key][col] = df[col].astype(str)
            dict_guardado[key].loc[0,col] = str(dict_guardado[key].loc[0,col]).replace('nan', 'Null')

    dict_guardado[key].columns = nombres_de_columnas_de_pagina_web

In [10]:
# Guardar los datos de corregidos de la sonda en un archivo CSV
rutas_de_archivos_csv_corregidos = []
os.makedirs(ruta_a_la_carpeta_de_guardado, exist_ok=True)
for serial, datos in dict_guardado.items():
    ruta_a_guardar = os.path.join(ruta_a_la_carpeta_de_guardado, f"datos_Localizacion_{serial}.csv")
    datos.to_csv(ruta_a_guardar, index=False)
    rutas_de_archivos_csv_corregidos.append(ruta_a_guardar)

In [11]:
# diccionario_de_datos_de_sondas.keys()

In [12]:
# 8
diccionario_de_sondas_en_fechas = carga.seleccionar_rango_de_fechas(diccionario = diccionario_de_datos_de_sondas, buscar_fechas_anteriores_al_estudio = False, config = config)

c:\programacion\codigos_python\drift_buoys\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [13]:
# 9
datos_de_sondas_sin_duplicados = carga.buscar_y_eliminar_duplicados(diccionario_de_sondas_en_fechas)

In [14]:
# 10
datos_ordenados = carga.ordernar_datos_por_fecha(diccionario_de_sondas_en_fechas)

Datos ordenados por fecha para la sonda B048-9869012.
Datos ordenados por fecha para la sonda B072-5339652.
Datos ordenados por fecha para la sonda B071-5339826.
Datos ordenados por fecha para la sonda B073-5339822.
Datos ordenados por fecha para la sonda B079-5339955.


In [15]:
# datos_ordenados.keys()

In [16]:
# Muchos de  los códigos de graficado buscan la variable tspan_rounded
# Se agregó con el mismo valor de tspan_de_envío
for serial in seriales_encontrados:
    if serial in datos_ordenados:
        datos_ordenados[serial]["tspan_rounded"] = datos_ordenados[serial]["tspan_de_envio"]


In [17]:
# Eliminar datos espurios (solo se revisa si hay valores de rapidez superiores a 2 m/s y se elimina toda la fila)
datos_finales = eliminar_datos_espurios(datos_ordenados)

In [18]:
# Agregar componentes de la velocidad al diccionario con los dataframe de cada sonda
datos_finales = carga.agregar_componentes_de_la_velocidad(datos_finales)

In [19]:
# datos_finales["4912208"].columns

In [20]:
# Guardar datos del estudio en archivo .pkl
carpeta_de_destino = ruta_a_la_carpeta_de_guardado
nombre_de_archivo = config.nombre_del_archivo_de_datos_procesados
guardar_diccionario_como_pickle(diccionario = datos_finales, 
                                ruta = carpeta_de_destino, 
                                nombre_archivo=nombre_de_archivo)


Diccionario guardado correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\despliegue_2\202610\datos_procesados_sondas_oceanograficas.pkl


In [21]:
# 15
tabla_de_porcentajes = calcular_porcentaje_de_datos_recibidos(datos_finales, config)

c:\programacion\codigos_python\drift_buoys\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [22]:
# tabla_de_porcentajes

In [23]:
# Guardar Tabla de porcentajes
guardar_porcentajes_en_excel(data= tabla_de_porcentajes, ruta= ruta_a_la_carpeta_de_guardado, nombre_de_archivo=config.nombre_del_excel_de_porcentajes)

Archivo Excel guardado correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\despliegue_2\202610\porcentajes_de_las_sondas.xlsx


In [24]:
# 17
graficar_series_y_guardar(
    datos= datos_finales,
    ruta_a_carpeta_de_guardado=ruta_a_la_carpeta_de_guardado, 
    mostrar_figura=False, 
    config=config
)

Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\despliegue_2\202610\transmision_B048-9869012.png
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\despliegue_2\202610\transmision_B072-5339652.png
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\despliegue_2\202610\transmision_B071-5339826.png
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\despliegue_2\202610\transmision_B073-5339822.png
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\despliegue_2\202610\transmision_B079-5339955.png


In [25]:
# 18
graficar_mapa_de_posiciones(
    datos=datos_finales, 
    ruta_a_la_carpeta_de_guardado = ruta_a_la_carpeta_de_guardado,
    mostrar_figura=False,
    config=config)

c:\programacion\codigos_python\drift_buoys\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


sonda B048-9869012 lat ini= 19.284428, lon ini= -91.974063
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\despliegue_2\202610\mapa_trayectoria_B048-9869012.png
sonda B072-5339652 lat ini= 19.29917, lon ini= -92.000756
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\despliegue_2\202610\mapa_trayectoria_B072-5339652.png
sonda B071-5339826 lat ini= 19.196388, lon ini= -91.863449
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\despliegue_2\202610\mapa_trayectoria_B071-5339826.png
sonda B073-5339822 lat ini= 19.179962, lon ini= -91.841347
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\despliegue_2\202610\mapa_trayectoria_B073-5339822.png
sonda B079-5339955 lat ini= 19.227372, lon ini= -91.904798
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\despliegue_2\202610\mapa_trayectoria_B079-5339955.

In [26]:
# 19. Guarda las pruebas de laboratorio en un archivo png

csv_a_png(rutas_de_archivos_csv_corregidos[0],
     carpeta_salida= ruta_a_la_carpeta_de_guardado,
     max_filas= 20,
     nombre_salida= f"mensaje_{seriales_encontrados[0]}.png"
)

print(f"Se ha guardado el png del mensaje de transmision")

Se ha guardado el png del mensaje de transmision
